#Initializations

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import StringType,DateType

#Reading Bronze Table

In [0]:
df = spark.table("workspace.bronze.crm_prd_info")

In [0]:
df.limit(10).display()

#Transformations

##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))
     


##Product Key Parsing

In [0]:

df = df.withColumn("cat_id", regexp_replace(substring(col("prd_key"), 1, 5), "-", "_"))
df = df.withColumn("prd_key", substring(col("prd_key"), 7, length(col("prd_key"))))

##Cost Cleanup

In [0]:
df = df.withColumn("prd_cost", coalesce(col("prd_cost"), lit(0)))

##Product Line Normalizations

In [0]:

df = (
    df
    # Normalize product line
    .withColumn(
        "prd_line",
        when(upper(col("prd_line")) == "M", "Mountain")
         .when(upper(col("prd_line")) == "R", "Road")
         .when(upper(col("prd_line")) == "S", "Other Sales")
         .when(upper(col("prd_line")) == "T", "Touring")
         .otherwise("n/a")
    )
)

##Date Casting

In [0]:

df = df.withColumn("prd_start_dt", col("prd_start_dt").cast(DateType()))

##Renaming Columns Name

In [0]:
RENAME_MAP = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

#Sanity Checks Of Dataframe

In [0]:

df.limit(10).display()

#Write Silver Table

In [0]:
df.write.mode("overwrite").saveAsTable("silver.crm_products")